# GumiNet — tutorial CPU / GPU opsional
Baseline riset eksperimental; data sintetis di bawah hanya memverifikasi alur, bukan akurasi dunia nyata.
Di Colab gratis pilih CPU atau GPU. Untuk dataset sendiri, ubah `DATASET` ke folder train/val/test.


In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
if importlib.util.find_spec("guminet") is None:
    checkout = Path("/content/GumiNet") if Path("/content").exists() else Path("GumiNet")
    if not checkout.exists():
        subprocess.run(["git", "clone", "https://github.com/Edong098/GumiNet.git", str(checkout)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(checkout)+"[export]"], check=True)
# Clone di atas membutuhkan source proyek ini sudah tersedia di GitHub; tidak ada push otomatis.


In [ ]:
import tempfile
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from guminet import create_model, GumiTrainer, GumiPredictor
from guminet.data import make_loaders
from guminet.utils import seed_everything
seed_everything(42)
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
workspace = Path(tempfile.mkdtemp(prefix="guminet-colab-"))
print("Device:", DEVICE, "workspace:", workspace)
model = create_model("guminet_nano", num_classes=3)


## Dataset kecil yang dapat dipelajari; pembacaan gambar tetap lazy

In [ ]:
DATASET = workspace / "dataset"  # Ganti dengan dataset Anda bila diperlukan.
rng = np.random.default_rng(42)
for split in ("train", "val", "test"):
    for label in range(3):
        folder = DATASET / split / f"class_{label}"
        folder.mkdir(parents=True, exist_ok=True)
        for i in range(4):
            pixels = rng.integers(0, 30, (40, 40, 3), dtype=np.uint8)
            pixels[:, :, label] += 180
            Image.fromarray(pixels).save(folder / f"{i}.png")
loaders = make_loaders(DATASET, input_size=128, batch_size=4)
print({k: v.dataset.summary() for k, v in loaders.items()})


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for label, ax in enumerate(axes):
    with Image.open(DATASET / "train" / f"class_{label}" / "0.png") as image:
        ax.imshow(image)
    ax.set_title(f"class_{label}"); ax.axis("off")
plt.show()


## Training, validasi, dan grafik

In [ ]:
config = dict(epochs=1, input_size=128, device=DEVICE, output_dir=str(workspace / "training"),
              scheduler="none", seed=42, progress=True)
trainer = GumiTrainer(model, config, loaders["train"].dataset.class_to_idx)
history = trainer.fit(loaders["train"], loaders["val"])
print("Validation macro F1:", history[-1]["val_macro_f1"])


In [ ]:
from guminet.evaluation import plot_history, evaluate
plot_history(history, workspace / "plots")
fig, ax = plt.subplots()
ax.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="train")
ax.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="val")
ax.legend(); ax.set_xlabel("Epoch"); plt.show()
from guminet.engine.checkpoint import model_from_checkpoint
checkpoint = workspace / "training/best.pt"
best_model, _ = model_from_checkpoint(checkpoint)
report = evaluate(best_model, loaders["test"], DEVICE, output_dir=workspace / "evaluation")
print("Synthetic test only:", report["accuracy"], report["macro_f1"])


## Simpan ke Google Drive (hanya Colab), lalu resume dari last checkpoint

In [ ]:
import shutil
try:
    from google.colab import drive
except ImportError:
    print("Di luar Colab: checkpoint tetap tersimpan lokal", checkpoint)
else:
    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/GumiNet")
    target.mkdir(parents=True, exist_ok=True)
    for name in ("best.pt", "last.pt"):
        shutil.copy2(workspace / "training" / name, target / name)


In [ ]:
resumed_model = create_model("guminet_nano", num_classes=3)
resumed = GumiTrainer(resumed_model, dict(config, epochs=2), loaders["train"].dataset.class_to_idx)
resumed.resume(workspace / "training/last.pt")
resumed.fit(loaders["train"], loaders["val"])
assert len(resumed.history) == 2


## Benchmark, prediksi, dan ONNX

In [ ]:
from guminet.benchmark import GumiBenchmark
benchmark = GumiBenchmark(best_model, "cpu").run(input_size=128, warmup=1, repeats=3, train_steps=1,
                                                output_dir=workspace / "benchmark", checkpoint=checkpoint)
print(benchmark)
predictor = GumiPredictor.from_checkpoint(checkpoint, DEVICE)
print(predictor.predict(DATASET / "test/class_0/0.png"))


In [ ]:
from guminet.export import export_onnx
export_report = export_onnx(checkpoint, workspace / "guminet.onnx")
assert export_report["validated"]
print(export_report)


## Berikutnya
Gunakan dataset nyata, pisahkan train/val/test sebelum tuning, dan ulangi eksperimen dengan beberapa seed.
Tidak ada pretrained weights resmi. Hasil notebook bukan bukti keunggulan V2 atau kesiapan deployment mobile.
